
# Fronteira de Pareto de Referência Empírica 8D via RSM + NSGA-III

Este notebook lê o arquivo `VRF_artigo.xlsx`, ajusta uma superfície de resposta quadrática para 8 respostas e usa `pymoo` com NSGA-III para gerar uma fronteira não-dominada empírica densa, chamada aqui de $P^*$.

**Arquitetura adotada para 16 GB de RAM:**

- O modelo RSM é um `Pipeline` do `scikit-learn` com `PolynomialFeatures(grau=2)` + `LinearRegression` multi-saída. Isso equivale a ajustar 8 regressões quadráticas independentes, mas com menos overhead.
- A otimização usa NSGA-III com direções de referência de Das-Dennis em 8 objetivos.
- O `save_history=False` é obrigatório: o notebook **não guarda o histórico das gerações**, apenas a população final.
- A paralelização usa `ThreadPool` com 12 threads, e não `multiprocessing.Pool`, para evitar cópias do modelo RSM em processos diferentes no Windows/Jupyter. Isso reduz pressão sobre os 16 GB de RAM.
- A visualização amostra pontos apenas para o gráfico, mas o CSV exporta o $P^*$ completo.

**Observação sobre maldição da dimensionalidade:** em 8 objetivos, o número de direções de referência explode combinatoriamente conforme `n_partitions` aumenta. Por isso, o perfil padrão é agressivo, mas ainda razoável para 16 GB.


In [ ]:

# ============================================================
# CÉLULA 1 — Instalação opcional
# ============================================================
# Rode esta célula apenas se algum pacote estiver faltando.
# Em ambiente já configurado, pule para a próxima célula.

# %pip install -U pandas numpy scikit-learn pymoo plotly openpyxl


In [ ]:

# ============================================================
# CÉLULA 2 — Imports e configurações gerais
# ============================================================

import os
import gc
from pathlib import Path
from math import comb
from multiprocessing.pool import ThreadPool

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from pymoo.core.problem import ElementwiseProblem, StarmapParallelization
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

ARQUIVO_EXCEL = Path(r"<CAMINHO_LOCAL_REMOVIDO>")

# Colunas do problema
DECISION_COLS = ["cs", "f", "md"]
OBJECTIVE_COLS = ["T", "MTTF", "WR", "Ra", "Rt", "Kp", "ROI", "OEE"]

# Limites contínuos codificados dos 3 fatores
# Raio axial rotacional do CCD para k = 3: alpha = 2^(3/4) = 1.6817928...
# (padronizado com o RSM-CNBI_orcamento; antes havia 1.682 aqui e
#  1.681793 lá — diferença imaterial, mas agora a constante é única e exata)
ALPHA = 2 ** 0.75
XL = -ALPHA * np.ones(len(DECISION_COLS), dtype=float)
XU = ALPHA * np.ones(len(DECISION_COLS), dtype=float)

# CORREÇÃO CRÍTICA DE REPRODUTIBILIDADE: a versão anterior atribuía chaves
# a um dicionário OBJECTIVE_SENSE que nunca era inicializado (NameError em
# kernel limpo) e omitia as quatro respostas de minimização (KeyError na
# construção de OBJECTIVE_SIGNS). O dicionário completo, com as 8 respostas,
# é declarado de uma vez — qualquer erro aqui inverte o sentido de um
# objetivo e corrompe P* silenciosamente.
OBJECTIVE_SENSE = {
    "T":    "max",
    "MTTF": "max",
    "WR":   "min",
    "Ra":   "min",
    "Rt":   "min",
    "Kp":   "min",
    "ROI":  "max",
    "OEE":  "max",
}
assert set(OBJECTIVE_SENSE) == set(OBJECTIVE_COLS), \
    "OBJECTIVE_SENSE precisa cobrir exatamente as respostas de OBJECTIVE_COLS."


# Sinal usado internamente pelo pymoo:
# min: mantém y
# max: usa -y, pois pymoo sempre minimiza
OBJECTIVE_SIGNS = np.array([
    1.0 if OBJECTIVE_SENSE[obj].lower() == "min" else -1.0
    for obj in OBJECTIVE_COLS
], dtype=float)

# Hardware: i5-12400F = 6 cores / 12 threads
N_THREADS = 12

print(f"Arquivo Excel: {ARQUIVO_EXCEL}")
print(f"Threads configuradas: {N_THREADS}")
print("Sentido dos objetivos:")
for obj in OBJECTIVE_COLS:
    print(f"  {obj}: {OBJECTIVE_SENSE[obj]}")


In [ ]:

# ============================================================
# CÉLULA 3 — Leitura e validação do Excel
# ============================================================

df = pd.read_excel(ARQUIVO_EXCEL)

# Remove espaços acidentais nos nomes das colunas
# Ex.: " cs " -> "cs"
df.columns = [str(c).strip() for c in df.columns]

expected_cols = DECISION_COLS + OBJECTIVE_COLS
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Colunas ausentes no Excel: {missing}")

# Mantém apenas as colunas necessárias e força numérico
df = df[expected_cols].copy()
for c in expected_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if df.isna().any().any():
    print("Linhas com valores ausentes ou não numéricos:")
    display(df[df.isna().any(axis=1)])
    raise ValueError("Existem valores ausentes/não numéricos nas colunas necessárias.")

X_train = df[DECISION_COLS].to_numpy(dtype=float)
Y_train = df[OBJECTIVE_COLS].to_numpy(dtype=float)

print("Dimensão de X_train:", X_train.shape)
print("Dimensão de Y_train:", Y_train.shape)
display(df.head())


In [ ]:

# ============================================================
# CÉLULA 4 — Ajuste RSM quadrático multi-saída
# ============================================================
# A regressão LinearRegression multi-saída ajusta uma equação independente
# para cada resposta, compartilhando apenas a matriz de projeto quadrática.
# Isso é equivalente a ajustar 8 RSMs quadráticas, mas com menos overhead.

rsm_model = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("reg", LinearRegression())
])

rsm_model.fit(X_train, Y_train)

Y_hat = rsm_model.predict(X_train)

fit_rows = []
for j, obj in enumerate(OBJECTIVE_COLS):
    fit_rows.append({
        "Resposta": obj,
        "R2_treino": r2_score(Y_train[:, j], Y_hat[:, j]),
        "y_min_obs": np.min(Y_train[:, j]),
        "y_max_obs": np.max(Y_train[:, j]),
        "y_min_pred_treino": np.min(Y_hat[:, j]),
        "y_max_pred_treino": np.max(Y_hat[:, j]),
    })

df_fit = pd.DataFrame(fit_rows)
display(df_fit)

# Tabela de coeficientes das equações quadráticas
poly = rsm_model.named_steps["poly"]
reg = rsm_model.named_steps["reg"]
feature_names = poly.get_feature_names_out(DECISION_COLS)

df_coef = pd.DataFrame(reg.coef_, index=OBJECTIVE_COLS, columns=feature_names)
df_coef.insert(0, "intercepto", reg.intercept_)

display(df_coef)


In [ ]:

# ============================================================
# CÉLULA 5 — Funções auxiliares de predição e conversão de sentido
# ============================================================

def prever_respostas_originais(X, batch_size=100_000):
    """
    Prediz as 8 respostas na escala original do problema.
    Usa batches para evitar picos de memória se X for muito grande.
    """
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)

    partes = []
    for ini in range(0, len(X), batch_size):
        fim = min(ini + batch_size, len(X))
        partes.append(rsm_model.predict(X[ini:fim]))
    return np.vstack(partes)


def respostas_para_pymoo(F_original):
    """
    Converte respostas originais para o espaço de minimização do pymoo.
    Se uma resposta estiver marcada como max, ela entra como -resposta.
    """
    return np.asarray(F_original, dtype=float) * OBJECTIVE_SIGNS


def calcular_ideal_nadir_respostas_originais(F_original):
    """
    Calcula Ideal e Nadir na escala original das respostas.
    Para objetivos de minimização: ideal=min, nadir=max.
    Para objetivos de maximização: ideal=max, nadir=min.
    """
    F_original = np.asarray(F_original, dtype=float)
    ideal = []
    nadir = []

    for j, obj in enumerate(OBJECTIVE_COLS):
        sentido = OBJECTIVE_SENSE[obj].lower()
        if sentido == "max":
            ideal.append(np.max(F_original[:, j]))
            nadir.append(np.min(F_original[:, j]))
        else:
            ideal.append(np.min(F_original[:, j]))
            nadir.append(np.max(F_original[:, j]))

    return (
        pd.Series(ideal, index=OBJECTIVE_COLS, name="Ideal_Z_star"),
        pd.Series(nadir, index=OBJECTIVE_COLS, name="Nadir_Z_nad")
    )

# Teste rápido no centro do domínio
x0 = np.array([[0.0, 0.0, 0.0]])
print("Predição no centro [0,0,0]:")
display(pd.DataFrame(prever_respostas_originais(x0), columns=OBJECTIVE_COLS))


In [ ]:
# ============================================================
# CÉLULA 6 — Classe do problema many-objective no pymoo
# COM RESTRIÇÃO ESFÉRICA: x.T @ x <= alpha^2
# ============================================================

# ALPHA agora é definido na Célula 2 (2**0.75, exato) — não redefinir aqui.
RAIO2 = ALPHA ** 2

class RSMManyObjectiveProblem(ElementwiseProblem):
    def __init__(self, elementwise_runner=None):
        super().__init__(
            n_var=len(DECISION_COLS),
            n_obj=len(OBJECTIVE_COLS),
            n_ieq_constr=1,   # uma restrição: x^T x - alpha^2 <= 0
            xl=XL,
            xu=XU,
            elementwise_runner=elementwise_runner
        )

    def _evaluate(self, x, out, *args, **kwargs):
        x = np.asarray(x, dtype=float).reshape(1, -1)

        f_original = rsm_model.predict(x)[0]

        # PONTO EXATO PARA MUDAR MAXIMIZAÇÃO/MINIMIZAÇÃO:
        # se OBJECTIVE_SENSE[obj] == "max", OBJECTIVE_SIGNS[j] = -1,
        # logo o pymoo minimiza -f_original[j].
        f_pymoo = OBJECTIVE_SIGNS * f_original

        # Restrição esférica do espaço codificado:
        # pymoo usa G <= 0 como viável.
        # Então:
        # x^T x <= alpha^2
        # vira:
        # x^T x - alpha^2 <= 0
        g_esfera = np.sum(x[0]**2) - RAIO2

        out["F"] = f_pymoo
        out["G"] = np.array([g_esfera])

In [ ]:
# ============================================================
# CÉLULA 6b — CHECKPOINT: reaproveitar P* já gerada (pula o NSGA-III)
# ============================================================
# Se você já rodou este notebook uma vez (as ~8h), NÃO precisa rodar de
# novo: basta apontar para o CSV exportado pela Célula 11 na execução
# anterior (referencia_pareto_8D.csv, por padrão) e marcar o flag abaixo.
#
# CARREGAR_PSTAR_EXISTENTE = True  -> pula as Células 7, 8 e 9 (direções de
#     referência, execução do NSGA-III e extração da população final) e
#     reconstrói X_P / F_P_original / df_P diretamente do CSV salvo.
# CARREGAR_PSTAR_EXISTENTE = False -> comportamento original: roda o
#     NSGA-III do zero (use se ainda não existe P* salva, ou se mudou o
#     RSM, o domínio, ou qualquer parâmetro que afete a otimização).
#
# O restante do notebook (ideal/nadir, exportação, gráficos, verificação
# de adequação) funciona igual nos dois casos, pois todos dependem apenas
# de X_P e F_P_original, que ficam definidos de qualquer forma abaixo.

CARREGAR_PSTAR_EXISTENTE = True
CAMINHO_PSTAR_EXISTENTE = Path("referencia_pareto_8D.csv")

if CARREGAR_PSTAR_EXISTENTE:
    if not CAMINHO_PSTAR_EXISTENTE.exists():
        raise FileNotFoundError(
            f"CARREGAR_PSTAR_EXISTENTE=True mas o arquivo "
            f"'{CAMINHO_PSTAR_EXISTENTE}' não foi encontrado. Coloque o CSV "
            f"exportado na execução anterior na pasta deste notebook, ou "
            f"defina CARREGAR_PSTAR_EXISTENTE = False para gerar do zero."
        )

    df_P = pd.read_csv(CAMINHO_PSTAR_EXISTENTE, encoding="utf-8-sig")

    faltando = [c for c in DECISION_COLS + OBJECTIVE_COLS if c not in df_P.columns]
    if faltando:
        raise ValueError(
            f"O CSV carregado não tem as colunas esperadas: {faltando}. "
            f"Verifique se é de fato a exportação da Célula 11 deste problema."
        )

    X_P = df_P[DECISION_COLS].to_numpy(dtype=float)
    F_P_original = df_P[OBJECTIVE_COLS].to_numpy(dtype=float)

    # Sanidade rápida: X_P precisa respeitar a esfera do domínio codificado
    # (com folga numérica). Se isso falhar, o CSV carregado provavelmente
    # não é compatível com o ALPHA/domínio atual deste notebook.
    normas = np.linalg.norm(X_P, axis=1)
    n_fora = int((normas > ALPHA + 1e-6).sum())
    if n_fora > 0:
        print(f"AVISO: {n_fora} ponto(s) de X_P fora da esfera de raio "
              f"{ALPHA:.6f}. O CSV pode ter sido gerado com outro raio/ALPHA "
              f"— confira antes de usar downstream (ex.: no CNBI).")

    print(f"P* CARREGADA de '{CAMINHO_PSTAR_EXISTENTE}': {len(df_P):,} pontos.")
    print("NSGA-III NÃO será executado (Células 7-9 puladas).")
else:
    print("CARREGAR_PSTAR_EXISTENTE = False: o NSGA-III será executado do zero "
          "nas próximas células.")


In [ ]:
if not CARREGAR_PSTAR_EXISTENTE:

    # ============================================================
    # CÉLULA 7 — Direções de referência NSGA-III e perfil de execução
    # ============================================================
    # Das-Dennis em 8 objetivos cresce combinatoriamente:
    # número de direções = C(M + p - 1, p), com M=8 e p=n_partitions.
    # p=8  ->  6.435 direções/população
    # p=10 -> 19.448 direções/população
    # p=12 -> 50.388 direções/população  (pesado para 16 GB e muito pesado para sorting)

    M = len(OBJECTIVE_COLS)

    print("Crescimento das direções de referência em 8 objetivos:")
    for p in [4, 6, 8, 10, 12]:
        print(f"  n_partitions={p:2d} -> {comb(M + p - 1, p):6d} direções")

    # Perfil recomendado para 16 GB.
    # Use "seguro_16gb" primeiro. Depois, se estiver estável, tente "agressivo_16gb".
    PERFIL = "seguro_16gb"

    if PERFIL == "seguro_16gb":
        N_PARTITIONS = 8       # 6.435 indivíduos. Bom equilíbrio para 16 GB.
        N_GEN = 1200           # Alto número de gerações para densidade empírica.
    elif PERFIL == "agressivo_16gb":
        N_PARTITIONS = 10      # 19.448 indivíduos. Pode ficar lento/pesado.
        N_GEN = 800
    elif PERFIL == "turbo_risco_memoria":
        N_PARTITIONS = 12      # 50.388 indivíduos. Alto risco em 16 GB.
        N_GEN = 400
    else:
        raise ValueError("PERFIL inválido.")

    ref_dirs = get_reference_directions("das-dennis", M, n_partitions=N_PARTITIONS)
    POP_SIZE = len(ref_dirs)

    print(f"\nPerfil: {PERFIL}")
    print(f"n_partitions: {N_PARTITIONS}")
    print(f"pop_size/ref_dirs: {POP_SIZE}")
    print(f"n_gen: {N_GEN}")
    print(f"avaliações aproximadas: {POP_SIZE * N_GEN:,}")
else:
    print("Pulando definição de direções de referência (P* carregada do checkpoint).")


In [ ]:
if not CARREGAR_PSTAR_EXISTENTE:

    # ============================================================
    # CÉLULA 8 — Execução do NSGA-III sem salvar histórico
    # ============================================================
    # Ponto crucial de RAM:
    # - save_history=False impede armazenar a população de cada geração.
    # - copy_algorithm=False evita cópias desnecessárias do objeto algoritmo.
    # - Não usamos callback acumulando arrays por geração.
    # - Mantemos apenas res.pop, isto é, a população final.

    pool = ThreadPool(N_THREADS)
    runner = StarmapParallelization(pool.starmap)

    problem = RSMManyObjectiveProblem(elementwise_runner=runner)

    algorithm = NSGA3(
        ref_dirs=ref_dirs,
        pop_size=POP_SIZE,
        eliminate_duplicates=True
    )

    termination = get_termination("n_gen", N_GEN)

    try:
        res = minimize(
            problem,
            algorithm,
            termination,
            seed=SEED,
            verbose=True,
            save_history=False,      # obrigatório para não estourar RAM
            copy_algorithm=False     # evita cópia extra do algoritmo/população
        )
    finally:
        pool.close()
        pool.join()

    print("\nOtimização finalizada.")
    print("res.X shape:", None if res.X is None else res.X.shape)
    print("res.F shape:", None if res.F is None else res.F.shape)

    # Limpeza preventiva
    gc.collect()
else:
    print("Pulando execução do NSGA-III (P* carregada do checkpoint).")


In [ ]:
if not CARREGAR_PSTAR_EXISTENTE:

    # ============================================================
    # CÉLULA 9 — Extração da população final e filtro não-dominado global
    # ============================================================
    # Mesmo que o pymoo já retorne soluções não-dominadas em res.X/res.F,
    # aqui refazemos o filtro sobre a população final para deixar o procedimento explícito.

    if res.pop is not None:
        X_final = np.asarray(res.pop.get("X"), dtype=float)
        F_final_pymoo = np.asarray(res.pop.get("F"), dtype=float)
    elif res.X is not None and res.F is not None:
        X_final = np.asarray(res.X, dtype=float)
        F_final_pymoo = np.asarray(res.F, dtype=float)
    else:
        raise RuntimeError("Não foi possível recuperar a população final do resultado do pymoo.")

    print("População final:", X_final.shape, F_final_pymoo.shape)

    # Remove duplicatas numéricas em X, sem destruir a densidade útil
    X_round = np.round(X_final, 10)
    _, idx_unique = np.unique(X_round, axis=0, return_index=True)
    idx_unique = np.sort(idx_unique)

    X_unique = X_final[idx_unique]
    F_unique_pymoo = F_final_pymoo[idx_unique]

    print("Após remover duplicatas em X:", X_unique.shape, F_unique_pymoo.shape)

    # Filtro não-dominado no espaço de minimização do pymoo
    idx_nd = NonDominatedSorting().do(F_unique_pymoo, only_non_dominated_front=True)

    X_P = X_unique[idx_nd]
    F_P_pymoo = F_unique_pymoo[idx_nd]

    # Recalcula as respostas originais para exportação e interpretação
    F_P_original = prever_respostas_originais(X_P)

    print("\nP* extraída:")
    print("X_P shape:", X_P.shape)
    print("F_P_original shape:", F_P_original.shape)
else:
    print("Pulando extração da população final (X_P e F_P_original já vieram do checkpoint).")
    print("X_P shape:", X_P.shape)
    print("F_P_original shape:", F_P_original.shape)


In [ ]:

# ============================================================
# CÉLULA 10 — Cálculo e print do Ideal e Nadir de P*
# ============================================================

Z_star, Z_nad = calcular_ideal_nadir_respostas_originais(F_P_original)

print("=" * 80)
print("PONTO IDEAL Z* — melhor valor alcançado em P* para cada resposta")
print("=" * 80)
print(Z_star.to_string(float_format=lambda x: f"{x:.10g}"))

print("\n" + "=" * 80)
print("PONTO NADIR Z^nad — pior valor alcançado em P* para cada resposta")
print("=" * 80)
print(Z_nad.to_string(float_format=lambda x: f"{x:.10g}"))

# Tabela compacta para visualização
extremos = pd.DataFrame({
    "sentido": [OBJECTIVE_SENSE[obj] for obj in OBJECTIVE_COLS],
    "Ideal_Z_star": Z_star,
    "Nadir_Z_nad": Z_nad,
    "amplitude_em_P_star": np.abs(Z_nad - Z_star)
})

display(extremos)


In [ ]:

# ============================================================
# CÉLULA 11 — Exportação de P* para CSV
# ============================================================
# O arquivo contém as 3 variáveis de decisão e as 8 respostas previstas
# na escala original do problema.

saida_csv = Path("referencia_pareto_8D.csv")

if CARREGAR_PSTAR_EXISTENTE and saida_csv.resolve() == CAMINHO_PSTAR_EXISTENTE.resolve():
    print(f"P* foi carregada de '{saida_csv}' neste modo checkpoint — "
          f"reexportando o mesmo conteúdo (idempotente, nada muda).")

P_dec = pd.DataFrame(X_P, columns=DECISION_COLS)
P_obj = pd.DataFrame(F_P_original, columns=OBJECTIVE_COLS)

df_P = pd.concat([P_dec, P_obj], axis=1)

df_P.to_csv(saida_csv, index=False, encoding="utf-8-sig")

print(f"Arquivo salvo: {saida_csv.resolve()}")
print(f"Número de pontos exportados em P*: {len(df_P):,}")
display(df_P.head())


In [ ]:

# ============================================================
# CÉLULA 12 — Coordenadas Paralelas de P*
# ============================================================
# Para não travar o navegador com milhares/dezenas de milhares de linhas,
# o gráfico usa uma amostra quando P* for muito grande.
# O CSV continua exportando todos os pontos.

try:
    import plotly.express as px

    N_VISUAL = 10_000
    if len(df_P) > N_VISUAL:
        df_vis = df_P.sample(N_VISUAL, random_state=SEED).copy()
        print(f"P* tem {len(df_P):,} pontos. Plotando amostra de {N_VISUAL:,} pontos.")
    else:
        df_vis = df_P.copy()
        print(f"Plotando todos os {len(df_vis):,} pontos de P*.")

    # Cor apenas para ajudar leitura visual. Troque para outra resposta se preferir.
    color_col = OBJECTIVE_COLS[0]

    fig = px.parallel_coordinates(
        df_vis,
        dimensions=OBJECTIVE_COLS,
        color=color_col,
        labels={c: c for c in OBJECTIVE_COLS},
        title="Coordenadas Paralelas — Fronteira de Pareto de Referência Empírica P*"
    )

    fig.update_layout(
        width=1200,
        height=650,
        font=dict(size=12)
    )

    fig.show()

except ImportError:
    print("Plotly não está instalado. Rode: %pip install plotly")


In [ ]:
# ============================================================
# CELL 13 — Parallel coordinates
# 0 = best and 1 = worst for all objectives
# Lines are colored according to T performance
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

N_VISUAL = 6435

if len(df_P) > N_VISUAL:
    df_vis_mpl = df_P.sample(
        N_VISUAL,
        random_state=SEED
    ).copy()
else:
    df_vis_mpl = df_P.copy()

# ------------------------------------------------------------
# Normalize objectives according to their optimization direction
# ------------------------------------------------------------
Y = df_vis_mpl[OBJECTIVE_COLS].to_numpy(dtype=float)

y_min = np.nanmin(Y, axis=0)
y_max = np.nanmax(Y, axis=0)

Y_norm = np.empty_like(Y, dtype=float)

for j, objective in enumerate(OBJECTIVE_COLS):
    value_range = y_max[j] - y_min[j] + 1e-12

    if OBJECTIVE_SENSE[objective].lower() == "min":
        # Minimum value is best
        Y_norm[:, j] = (Y[:, j] - y_min[j]) / value_range
    else:
        # Maximum value is best
        Y_norm[:, j] = (y_max[j] - Y[:, j]) / value_range

# ------------------------------------------------------------
# Normalize T: 0 = best and 1 = worst
# T is a maximization objective
# ------------------------------------------------------------
T = df_vis_mpl["T"].to_numpy(dtype=float)

t_min = np.nanmin(T)
t_max = np.nanmax(T)

T_norm = (t_max - T) / (t_max - t_min + 1e-12)

# Blue = better T
# Red = worse T
cmap = plt.get_cmap("RdBu_r")
color_norm = Normalize(vmin=0.0, vmax=1.0)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 6))

for i in range(Y_norm.shape[0]):
    ax.plot(
        range(len(OBJECTIVE_COLS)),
        Y_norm[i, :],
        color=cmap(color_norm(T_norm[i])),
        alpha=0.15,
        linewidth=0.7
    )

ax.set_xticks(range(len(OBJECTIVE_COLS)))
ax.set_xticklabels(
    OBJECTIVE_COLS,
    rotation=30,
    ha="right"
)

ax.set_ylim(0, 1)
ax.set_ylabel("Normalized performance — 0: best | 1: worst")
ax.grid(True, alpha=0.25)

# Color bar
sm = ScalarMappable(norm=color_norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Normalized T performance — 0: best | 1: worst")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VERIFICAÇÃO DE CONSISTÊNCIA ENTRE MODELOS RSM
# (rodar ANTES de interpretar a % de pontos "fora de P*")
# ============================================================
# CONTEXTO: o RSM-CNBI_orcamento.ipynb ajusta seu PRÓPRIO modelo RSM (OLS
# matricial manual) a partir do mesmo VRF_artigo.xlsx que este notebook usa
# (Pipeline sklearn). Ambos são OLS de quadrática completa sobre os MESMOS
# dados — em teoria, coeficientes idênticos. Mas "em teoria" não substitui
# checar: se os dois RSMs divergirem (por qualquer motivo — filtragem de
# linhas diferente, ordem de termos, etc.), os pontos do CNBI podem parecer
# "não dominados por P*" só porque estão sendo avaliados por um modelo
# ligeiramente diferente, e não porque exploram uma região genuinamente
# nova da fronteira.
#
# Este teste é BARATO e DECISIVO: pega uma amostra dos X (variáveis de
# decisão) exportados pelo CNBI, avalia esses MESMOS x através do RSM
# deste notebook (rsm_model, treinado independentemente) e compara com os
# valores "_fisico" que o CNBI já exportou (calculados pelo RSM dele).
#
# Se baterem (diferença relativa ~0): os modelos são consistentes, e a alta
# % de pontos "fora de P*" é explicada por resistência de dominância em
# muitos objetivos (Liu et al., 2023, já citado na Seção 3.9) — P* cobre
# densamente a variedade de Pareto, mas discretiza um número finito de
# direções de trade-off; outra direção, mesmo pior "no geral", pode não ser
# dominada por nenhum ponto específico da amostra. Isso é uma propriedade
# esperada de espaços m=8, não um defeito do pipeline.
#
# Se NÃO baterem: há uma inconsistência de modelo entre os notebooks que
# precisa ser corrigida ANTES de confiar em qualquer número de GD/IGD/HV —
# problema mais sério do que a escolha de referência (LOO vs. ingênua).

from pathlib import Path

N_AMOSTRA_VERIFICACAO = 20
TOL_REL_VERIFICACAO = 1e-3   # 0.1% de diferença relativa é o limiar de alerta

arquivos_cnbi = sorted(Path(".").glob("fronteira_ND_cnbi*.csv"))

if not arquivos_cnbi:
    print("Nenhum arquivo fronteira_ND_cnbi*.csv encontrado — verificação "
          "de consistência pulada. Rode isto depois de gerar a fronteira "
          "CNBI, antes de interpretar a checagem de adequação de P*.")
else:
    arq = arquivos_cnbi[0]
    d_cnbi = pd.read_csv(arq)

    cols_x = [c for c in DECISION_COLS if c in d_cnbi.columns]
    cols_f_fisico = [f"{c}_fisico" for c in OBJECTIVE_COLS]

    faltando_x = [c for c in DECISION_COLS if c not in d_cnbi.columns]
    faltando_f = [c for c in cols_f_fisico if c not in d_cnbi.columns]

    if faltando_x or faltando_f:
        print(f"'{arq.name}' não tem as colunas esperadas "
              f"(faltam X: {faltando_x}, faltam F: {faltando_f}) — "
              f"verificação pulada.")
    else:
        n_amostra = min(N_AMOSTRA_VERIFICACAO, len(d_cnbi))
        amostra = d_cnbi.sample(n_amostra, random_state=42)

        X_amostra = amostra[DECISION_COLS].to_numpy(dtype=float)
        F_cnbi_exportado = amostra[cols_f_fisico].to_numpy(dtype=float)

        # Reavalia os MESMOS x através do RSM deste notebook (ajustado
        # independentemente, mesma fonte de dados)
        F_pstar_rsm = prever_respostas_originais(X_amostra)

        diff_abs = np.abs(F_pstar_rsm - F_cnbi_exportado)
        escala = np.maximum(np.abs(F_cnbi_exportado), 1e-8)
        diff_rel = diff_abs / escala

        df_verif = pd.DataFrame({
            "resposta": OBJECTIVE_COLS,
            "diff_abs_media": diff_abs.mean(axis=0),
            "diff_abs_max": diff_abs.max(axis=0),
            "diff_rel_media": diff_rel.mean(axis=0),
            "diff_rel_max": diff_rel.max(axis=0),
        })

        print(f"Comparando {n_amostra} pontos de '{arq.name}' "
              f"avaliados pelos DOIS RSMs (CNBI-exportado vs. P*-deste-notebook):")
        display(df_verif.style.format({
            "diff_abs_media": "{:.6g}", "diff_abs_max": "{:.6g}",
            "diff_rel_media": "{:.4%}", "diff_rel_max": "{:.4%}",
        }))

        pior_rel = df_verif["diff_rel_max"].max()
        if pior_rel <= TOL_REL_VERIFICACAO:
            print(f"\n=> Modelos CONSISTENTES (maior diferença relativa "
                  f"{pior_rel:.4%} <= {TOL_REL_VERIFICACAO:.1%}). A alta % "
                  f"de pontos \"fora de P*\" na checagem de adequação abaixo "
                  f"reflete resistência de dominância em muitos objetivos, "
                  f"não erro de modelo — pode ser reportada como achado "
                  f"(Seção 3.9/7), sujeita à leitura da tabela por fonte.")
        else:
            print(f"\n=> ALERTA: diferença relativa de até {pior_rel:.2%} "
                  f"entre os dois RSMs, acima do limiar de {TOL_REL_VERIFICACAO:.1%}. "
                  f"Antes de interpretar qualquer % \"fora de P*\" ou recalcular "
                  f"GD/IGD/HV, investigue a origem da divergência: linhas do "
                  f"Excel filtradas diferentemente entre os notebooks, ordem "
                  f"de colunas nas matrizes de projeto, ou precisão numérica "
                  f"do ajuste (inv vs. pinv). Isso é mais urgente que a "
                  f"escolha de referência (LOO vs. ingênua).")


In [ ]:
# ============================================================
# ADEQUAÇÃO DE P* — POR MAGNITUDE (distância normalizada + indicador ε),
# NÃO por contagem/percentual de sobreviventes não-dominados
# ============================================================
# Por que não usar % de sobreviventes ND como critério:
# em m=8 objetivos, a dominância de Pareto perde poder discriminatório
# (Liu et al., 2023, já citado na Seção 3.9) — mesmo um método MEDÍOCRE
# tende a ter uma fração grande de pontos "não-dominados" em relação a
# qualquer referência densa, simplesmente porque dominar exige vencer
# simultaneamente em TODOS os 8 critérios. Um percentual alto de
# sobreviventes é portanto uma consequência estrutural da dimensão, não
# um diagnóstico de que P* é inadequada. Contar sobreviventes responde
# "quantos pontos não são estritamente piores em tudo?" — a pergunta
# certa é "por QUANTO esses pontos escapam, e isso importa perto da
# escala das diferenças que estamos usando para comparar os métodos?".
#
# P* permanece INDEPENDENTE em todos os cálculos abaixo — nunca é
# aumentada, fundida ou realimentada com pontos de nenhum método. Ela é
# só o objeto que está sendo avaliado.
#
# Três medidas de magnitude, todas no espaço normalizado por
# utopia/nadir de P* (mesma normalização usada no comparacao_fronteiras,
# então os números aqui são diretamente comparáveis às diferenças de
# GD/IGD já reportadas):
#
# 1) DISTÂNCIA NORMALIZADA — para cada ponto de um método não dominado
#    por P*, a distância euclidiana ao ponto mais próximo de P* (é,
#    literalmente, a contribuição individual desse ponto ao GD do
#    método). Pequena = o ponto está "quase" em P*, a lacuna é
#    irrelevante. Grande = região genuinamente distante.
#
# 2) INDICADOR ε ADITIVO (Zitzler et al., 2003) — para cada ponto a de
#    um método, eps(a) = min_{p em P*} max_j (a_j - p_j). É o menor
#    relaxamento aditivo que, aplicado a P*, faria algum ponto de P*
#    dominar a. eps(a) <= 0 significa que a já é dominado (não deveria
#    sobrar no filtro ND da união); eps(a) pequeno e positivo (ex.: <
#    1% da amplitude) é uma vitória marginal em 1 objetivo, compatível
#    com ruído; eps(a) grande é uma vitória substantiva.
#
# 3) MAGNITUDE DA "MELHORIA" POR OBJETIVO — decompõe eps(a): em qual
#    objetivo específico o ponto vence P*, e por quanto.
#
# Critério de decisão: comparar a DISTRIBUIÇÃO dessas magnitudes (não a
# contagem) com a escala das diferenças de GD/IGD já usadas para
# comparar os métodos (ex.: CNBI 0,136 vs. VRF-NBI 0,389 -> diferença
# de referência ~0,25 em unidades normalizadas). Se a mediana/p90 de
# eps(a) e da distância normalizada ficarem MUITO abaixo dessa escala,
# a lacuna de P* é irrelevante para as conclusões já reportadas. Se
# ficarem na mesma ordem de grandeza ou maiores, a comparação precisa de
# mais cautela — mas a correção segue sendo a referência leave-one-out
# do comparacao_fronteiras.ipynb (Seção 3), NUNCA aumentar P* fundindo-a
# com pontos de método.

from pathlib import Path

ESCALA_REFERENCIA_GD_IGD = 0.25  # ajuste com a menor diferença de GD/IGD
                                  # que sustenta uma conclusão do trabalho
                                  # (ex.: |IGD_CNBI - IGD_VRF| já observado)


def _mascara_nd(F):
    F = np.asarray(F, dtype=float)
    n = F.shape[0]
    is_nd = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_nd[i]:
            continue
        dom = np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)
        dom[i] = False
        if dom.any():
            is_nd[i] = False
    return is_nd


def epsilon_aditivo_e_distancia(A_n, Pstar_n, chunk=200):
    """Para cada ponto de A_n (normalizado, minimização), retorna:
    - eps(a)  = min_p max_j (a_j - p_j)           [indicador ε aditivo]
    - dist(a) = min_p ||a - p||_2                  [distância normalizada]
    - j_vencedor(a) = objetivo onde a vitória de 'a' sobre o melhor p é máxima
    Processado em chunks para não estourar memória com |A|x|P*|x m.
    """
    n_obj = A_n.shape[1]
    eps = np.empty(len(A_n))
    dist = np.empty(len(A_n))
    j_vencedor = np.empty(len(A_n), dtype=int)

    for ini in range(0, len(A_n), chunk):
        fim = min(ini + chunk, len(A_n))
        Ac = A_n[ini:fim]                                   # (c, m)
        diffs = Ac[:, None, :] - Pstar_n[None, :, :]         # (c, |P*|, m)
        margem_por_p = diffs.max(axis=2)                    # (c, |P*|) = eps se 'p' fosse o dominador
        idx_melhor_p = margem_por_p.argmin(axis=1)           # melhor p por linha
        eps[ini:fim] = margem_por_p[np.arange(fim - ini), idx_melhor_p]

        dist_por_p = np.linalg.norm(diffs, axis=2)           # (c, |P*|)
        dist[ini:fim] = dist_por_p.min(axis=1)

        diff_melhor = diffs[np.arange(fim - ini), idx_melhor_p, :]  # (c, m)
        j_vencedor[ini:fim] = diff_melhor.argmin(axis=1)     # objetivo mais negativo = maior vitória

    return eps, dist, j_vencedor


fontes = []  # (nome, matriz m-objetivos em unidades FÍSICAS)
fontes.append(("P*", F_P_original))

# CNBI — saída do RSM-CNBI (colunas <resp>_fisico)
for arq in sorted(Path(".").glob("fronteira_ND_cnbi_*.csv")):
    d = pd.read_csv(arq)
    cols = [f"{c}_fisico" for c in OBJECTIVE_COLS]
    if all(c in d.columns for c in cols):
        fontes.append((f"CNBI [{arq.name}]", d[cols].to_numpy(dtype=float)))

# VRF-NBI
arq_vrf = Path("VRF_Pareto.xlsx")
if arq_vrf.exists():
    d = pd.read_excel(arq_vrf, header=0, usecols="A:R")
    if all(c in d.columns for c in OBJECTIVE_COLS):
        fontes.append(("VRF-NBI", d[OBJECTIVE_COLS].to_numpy(dtype=float)))

# NSGA-III / MOEA-D (seeds)
for prefixo in ("nsga3", "moead"):
    for arq in sorted(Path(".").glob(f"{prefixo}_seed*.csv")):
        d = pd.read_csv(arq)
        if all(c in d.columns for c in OBJECTIVE_COLS):
            fontes.append((f"{prefixo} [{arq.name}]", d[OBJECTIVE_COLS].to_numpy(dtype=float)))

if len(fontes) == 1:
    print("Nenhuma fronteira de método encontrada na pasta — verificação pulada.")
    print("Coloque aqui os CSVs (fronteira_ND_cnbi_*, VRF_Pareto.xlsx, "
          "nsga3_seed*, moead_seed*) e rode novamente.")
else:
    # ---- normalização por utopia/nadir de P* (mesma convenção de
    #      comparacao_fronteiras.ipynb -> números diretamente comparáveis) ----
    Pstar_min = F_P_original * OBJECTIVE_SIGNS
    ideal = Pstar_min.min(axis=0)
    nadir = Pstar_min.max(axis=0)
    amplitude = np.where(np.abs(nadir - ideal) < 1e-12, 1.0, nadir - ideal)

    def normalizar(F_fisico):
        return (np.asarray(F_fisico, dtype=float) * OBJECTIVE_SIGNS - ideal) / amplitude

    Pstar_n = normalizar(F_P_original)

    # ---- contexto estrutural: % ND (informativo, NÃO usado como critério) ----
    blocos, rotulos = [], []
    for nome, F in fontes:
        blocos.append(normalizar(F))
        rotulos.extend([nome] * len(F))
    F_uniao_n = np.vstack(blocos)
    rotulos = np.array(rotulos)
    nd = _mascara_nd(F_uniao_n)

    print("CONTEXTO ESTRUTURAL (não usar como critério de aceite — ver acima):")
    print(f"  União: {len(F_uniao_n)} pontos | não-dominados: {int(nd.sum())} "
          f"({100 * nd.sum() / len(F_uniao_n):.1f}% — esperado ser alto em m=8).\n")

    # ---- o que importa: magnitude por método ----
    linhas_magnitude = []
    detalhe_por_metodo = {}

    for nome, F in fontes:
        if nome == "P*":
            continue
        A_n = normalizar(F)
        eps, dist, j_vencedor = epsilon_aditivo_e_distancia(A_n, Pstar_n)

        # só o subconjunto que sobrevive ND na união é "candidato a escape";
        # os demais já são dominados por P* (eps <= 0) e não preocupam.
        escapa = eps > 1e-9

        linhas_magnitude.append({
            "fonte": nome,
            "n_pontos": len(A_n),
            "n_escapam_Pstar": int(escapa.sum()),
            "eps_mediana": float(np.median(eps[escapa])) if escapa.any() else 0.0,
            "eps_p90": float(np.percentile(eps[escapa], 90)) if escapa.any() else 0.0,
            "eps_max": float(eps.max()) if len(eps) else 0.0,
            "dist_norm_mediana": float(np.median(dist[escapa])) if escapa.any() else 0.0,
            "dist_norm_p90": float(np.percentile(dist[escapa], 90)) if escapa.any() else 0.0,
            "dist_norm_max": float(dist.max()) if len(dist) else 0.0,
        })
        detalhe_por_metodo[nome] = {
            "eps": eps, "dist": dist, "j_vencedor": j_vencedor, "escapa": escapa,
        }

    df_magnitude_pstar = pd.DataFrame(linhas_magnitude)
    print("Magnitude do que cada método encontra fora de P* "
          "(unidades normalizadas por utopia/nadir de P*):")
    display(df_magnitude_pstar.style.format({
        c: "{:.4f}" for c in df_magnitude_pstar.columns if c.startswith(("eps_", "dist_norm_"))
    }))

    pior_eps_p90 = df_magnitude_pstar["eps_p90"].max() if len(df_magnitude_pstar) else 0.0
    pior_dist_p90 = df_magnitude_pstar["dist_norm_p90"].max() if len(df_magnitude_pstar) else 0.0

    print(f"\nEscala de referência (menor diferença de GD/IGD que sustenta uma "
          f"conclusão do trabalho): {ESCALA_REFERENCIA_GD_IGD:.3f}")
    print(f"Pior p90 do indicador ε entre os métodos: {pior_eps_p90:.4f}")
    print(f"Pior p90 da distância normalizada entre os métodos: {pior_dist_p90:.4f}")

    if max(pior_eps_p90, pior_dist_p90) < 0.1 * ESCALA_REFERENCIA_GD_IGD:
        print("\n=> Magnitude do que escapa de P* é pequena frente à escala das "
              "diferenças de GD/IGD já reportadas (< 10%). P* é adequada como "
              "referência independente para as conclusões atuais; a fração alta "
              "de sobreviventes ND é o efeito esperado de dominância em m=8, "
              "não uma falha de cobertura relevante.")
    elif max(pior_eps_p90, pior_dist_p90) < ESCALA_REFERENCIA_GD_IGD:
        print("\n=> Magnitude moderada: menor que a escala de referência, mas não "
              "desprezível. Reporte os indicadores ε/distância normalizada junto "
              "com GD/IGD como evidência de que a diferença entre métodos é "
              "robusta a essa lacuna de P*. Mantenha P* independente; use a "
              "referência leave-one-out do comparacao_fronteiras.ipynb (Seção 3) "
              "para os números finais, nunca uma P* aumentada.")
    else:
        print("\n=> Magnitude comparável ou maior que a escala de referência: "
              "pelo menos um método encontra pontos genuinamente distantes de "
              "P*, numa margem que poderia mudar uma conclusão. Ainda assim, a "
              "ação correta NÃO é aumentar P* (viés auto-referencial já "
              "discutido) — é reportar o indicador ε como métrica complementar "
              "de primeira classe (Zitzler et al., 2003) ao lado de GD/IGD, e "
              "usar a referência leave-one-out por método no comparacao_fronteiras.ipynb.")

    # ---- em qual objetivo específico a "vitória" ocorre (top 2 métodos) ----
    if len(df_magnitude_pstar):
        top2 = df_magnitude_pstar.nlargest(2, "eps_p90")["fonte"].tolist()
        for nome in top2:
            det = detalhe_por_metodo[nome]
            if not det["escapa"].any():
                continue
            objetivos_vencedores = np.array(OBJECTIVE_COLS)[det["j_vencedor"][det["escapa"]]]
            contagem_obj = pd.Series(objetivos_vencedores).value_counts()
            print(f"\nObjetivo em que '{nome}' mais supera P* (entre os pontos "
                  f"que escapam):")
            print(contagem_obj.to_string())

    # ---- exportação (auditoria — não usar como referência de GD/IGD) ----
    df_magnitude_pstar.to_csv("adequacao_pstar_magnitude.csv", index=False)
    print("\nTabela de magnitude salva em 'adequacao_pstar_magnitude.csv' "
          "(uso: contextualizar GD/IGD, não para reconstruir P*).")



## Observações finais

1. **Se a otimização ficar lenta:** reduza `N_GEN` antes de reduzir `N_PARTITIONS`. Reduzir `N_PARTITIONS` reduz diretamente a densidade da população final.
2. **Se a RAM começar a ficar crítica:** mantenha `save_history=False`, evite callbacks que armazenem arrays por geração e use `PERFIL = "seguro_16gb"`.
3. **Se quiser maximizar MTTF, ROI ou OEE:** altere `OBJECTIVE_SENSE` na célula 2. O notebook continuará exportando as respostas na escala original.
4. **Ponto Nadir empírico:** o Nadir calculado aqui é o pior valor por objetivo dentro da fronteira empírica $P^*$, não necessariamente o Nadir verdadeiro global do problema contínuo.
5. **RSM é surrogate:** a qualidade de $P^*$ depende diretamente da qualidade das superfícies quadráticas ajustadas a partir do DOE.
